In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio

import yaml

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

--------
# Sanity check: compare the no-magnification counts with original catalog

In [3]:
cat = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/data/lrg_xcorr/catalogs/dr9_extended_lrg_0.49.0_pzbins_20230120.fits'))
print(len(cat))

35354635


In [4]:
min_nobs = 2
# maskbits = [1, 8, 9, 11, 12, 13]

mask = (cat['PIXEL_NOBS_G']>=min_nobs) & (cat['PIXEL_NOBS_R']>=min_nobs) & (cat['PIXEL_NOBS_Z']>=min_nobs)
print(np.sum(~mask)/len(mask))

mask &= (cat['DEC']>-29)
print(np.sum(~mask)/len(mask))

# mask_clean = np.ones(len(cat), dtype=bool)
# for bit in maskbits:
#     mask_clean &= (cat['MASKBITS'] & 2**bit)==0
# print(np.sum(~mask_clean)/len(mask_clean))
mask_clean = cat['lrg_mask']==0
mask &= mask_clean

cat = cat[mask]
print(len(cat))

0.05160975357262209
0.20465025307148554
25196832


In [5]:
for photsys in ['N', 'S']:
    print(photsys)
    for bin_index in range(1, 5):
        mask = cat['PHOTSYS']==photsys
        mask &= cat['pz_bin']==bin_index
        print('bin', bin_index, np.sum(mask))
    print()

N
bin 1 823269
bin 2 1361448
bin 3 1920672
bin 4 1999582

S
bin 1 1835751
bin 2 3087540
bin 3 4122299
bin 4 4271123



In [6]:
counts_path = '/global/cfs/cdirs/desi/users/rongpu/lrg_xcorr/magnification/counts/extended_lrg_counts.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)

In [7]:
for photsys in ['N', 'S']:
    if photsys=='N':
        field = 'north'
    else:
        field = 'south'
    print(photsys)
    mask = cat['PHOTSYS']==photsys
    mask &= cat['extended_lrg'].copy()
    print('all', np.sum(mask), counts['{}_all_1.000'.format(field)], 'diff: {:.3f}%'.format((counts['{}_all_1.000'.format(field)]-np.sum(mask))/np.sum(mask)*100))
    for bin_index in range(1, 5):
        mask = cat['PHOTSYS']==photsys
        mask &= cat['pz_bin']==bin_index
        print('bin', bin_index, np.sum(mask), counts['{}_bin_{}_1.000'.format(field, bin_index)], 'diff: {:.3f}%'.format((counts['{}_bin_{}_1.000'.format(field, bin_index)]-np.sum(mask))/np.sum(mask)*100))
    print()

N
all 7459163 7459177 diff: 0.000%
bin 1 823269 823244 diff: -0.003%
bin 2 1361448 1361718 diff: 0.020%
bin 3 1920672 1920273 diff: -0.021%
bin 4 1999582 1999416 diff: -0.008%

S
all 16523203 16523233 diff: 0.000%
bin 1 1835751 1835626 diff: -0.007%
bin 2 3087540 3088038 diff: 0.016%
bin 3 4122299 4122179 diff: -0.003%
bin 4 4271123 4270768 diff: -0.008%



--------
# Number count slope

In [9]:
counts_path = '/global/cfs/cdirs/desi/users/rongpu/lrg_xcorr/magnification/counts/extended_lrg_counts.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 0.752 +- 0.007
bin 1:  s = 0.776 +- 0.022
bin 2:  s = 0.885 +- 0.017
bin 3:  s = 0.672 +- 0.014
bin 4:  s = 0.654 +- 0.014

south
all:  s = 0.748 +- 0.005
bin 1:  s = 0.736 +- 0.015
bin 2:  s = 0.844 +- 0.011
bin 3:  s = 0.699 +- 0.010
bin 4:  s = 0.656 +- 0.010

combined
all:  s = 0.750 +- 0.004
bin 1:  s = 0.748 +- 0.012
bin 2:  s = 0.857 +- 0.009
bin 3:  s = 0.690 +- 0.008
bin 4:  s = 0.655 +- 0.008



No fiberflux factor:

In [10]:
counts_path = '/global/cfs/cdirs/desi/users/rongpu/lrg_xcorr/magnification/counts/extended_lrg_counts_no_ff.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 0.838 +- 0.007
bin 1:  s = 0.777 +- 0.022
bin 2:  s = 0.899 +- 0.017
bin 3:  s = 0.740 +- 0.014
bin 4:  s = 0.817 +- 0.014

south
all:  s = 0.850 +- 0.005
bin 1:  s = 0.738 +- 0.015
bin 2:  s = 0.869 +- 0.011
bin 3:  s = 0.784 +- 0.010
bin 4:  s = 0.853 +- 0.010

combined
all:  s = 0.846 +- 0.004
bin 1:  s = 0.750 +- 0.012
bin 2:  s = 0.878 +- 0.009
bin 3:  s = 0.770 +- 0.008
bin 4:  s = 0.841 +- 0.008



No photo-z shift:

In [11]:
counts_path = '/global/cfs/cdirs/desi/users/rongpu/lrg_xcorr/magnification/counts/extended_lrg_counts_no_pz_mag.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 0.752 +- 0.007
bin 1:  s = 0.843 +- 0.022
bin 2:  s = 0.793 +- 0.017
bin 3:  s = 0.542 +- 0.014
bin 4:  s = 0.812 +- 0.014

south
all:  s = 0.748 +- 0.005
bin 1:  s = 0.837 +- 0.015
bin 2:  s = 0.789 +- 0.011
bin 3:  s = 0.562 +- 0.010
bin 4:  s = 0.813 +- 0.010

combined
all:  s = 0.750 +- 0.004
bin 1:  s = 0.839 +- 0.012
bin 2:  s = 0.790 +- 0.009
bin 3:  s = 0.556 +- 0.008
bin 4:  s = 0.812 +- 0.008

